# 9주차 정답 Notebook — 대규모 센서 데이터와 간단한 불량 예측

## 1단계. 데이터 불러오기

In [1]:
import pandas as pd

df = pd.read_csv("../../data/weekly/week09/week09_fab_beginner.csv")
df.head()

,SensorTime,Chamber_Temperature_edu,Chamber_Pressure_edu,Gas_Flow_edu,RF_Power_edu,Vacuum_Level_edu,Cooling_Water_Temperature_edu,Vibration_edu,Process_Time_edu,주요센서_A,주요센서_B,검사결과
0,2008-07-19 11:55:00,-1.73,-0.00,64.67,0.02,33.16,10.05,18.77,2.71,-5419.00,64.23,0
1,2008-07-19 12:32:00,0.81,-0.00,141.44,0.01,2.27,8.63,10.44,5.71,-5441.50,68.42,0
2,2008-07-19 13:17:00,23.82,-0.00,240.78,0.01,29.17,14.25,10.32,5.76,-5447.75,67.13,1
3,2008-07-19 14:43:00,24.38,-0.01,113.56,0.04,13.41,5.18,15.71,5.39,-5468.25,62.93,0
4,2008-07-19 15:22:00,-12.29,-0.00,148.07,0.02,10.74,11.41,12.76,2.01,-5476.25,62.83,0


## 2단계. 데이터 크기와 정상/이상 비율 확인하기

In [2]:
print(df.shape)
print(df["검사결과"].value_counts())

(1565, 12)
검사결과
0    1462
1     103
Name: count, dtype: int64


**답**: 데이터 불균형(class imbalance)이라고 부른다.

## 3단계. 결측값 확인 및 처리

In [3]:
feature_cols = [
    "Chamber_Temperature_edu", "Chamber_Pressure_edu", "Gas_Flow_edu", "RF_Power_edu",
    "Vacuum_Level_edu", "Cooling_Water_Temperature_edu", "Vibration_edu", "Process_Time_edu",
    "주요센서_A", "주요센서_B",
]
print(df[feature_cols].isna().sum())
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].mean())

Chamber_Temperature_edu           7
Chamber_Pressure_edu              2
Gas_Flow_edu                      2
RF_Power_edu                     24
Vacuum_Level_edu                  0
Cooling_Water_Temperature_edu     0
Vibration_edu                     0
Process_Time_edu                  0
주요센서_A                            0
주요센서_B                            0
dtype: int64


## 4단계. 학습 데이터와 테스트 데이터 나누기

In [4]:
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df["검사결과"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("학습 데이터:", X_train.shape, " 테스트 데이터:", X_test.shape)

학습 데이터: (1095, 10)  테스트 데이터: (470, 10)


## 5단계. 의사결정나무 모델 학습하기

In [5]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train, y_train)
print("학습 완료")

학습 완료


## 6단계. 예측하고 정확도 확인하기

In [6]:
from sklearn.metrics import accuracy_score

y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"정확도: {acc*100:.2f}%")

정확도: 91.49%


## 7단계. '모두 정상'이라고만 예측했을 때의 정확도와 비교하기

In [7]:
naive_acc = (y_test == 0).mean() * 100
print(f"'모두 정상'이라고만 예측했을 때 정확도: {naive_acc:.2f}%")

'모두 정상'이라고만 예측했을 때 정확도: 93.40%


**답**: 정확도만 보면 두 결과가 비슷해 보일 수 있다. 정확도만으로 모델이 좋다고 말할 수 없다 —
정상 데이터가 압도적으로 많아서 아무것도 배우지 않아도 정확도가 높게 나오기 때문이다.

## 8단계. 혼동행렬로 자세히 확인하기

In [8]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
print("혼동행렬 (행=실제, 열=예측, 순서 0=정상,1=이상):")
print(cm)

혼동행렬 (행=실제, 열=예측, 순서 0=정상,1=이상):
[[427  12]
 [ 28   3]]


## 9단계(도전). 어떤 센서가 예측에 가장 중요했는지 확인하기

In [9]:
importances = pd.Series(model.feature_importances_, index=feature_cols)
importances.sort_values(ascending=False)

주요센서_A                           0.220254
Chamber_Temperature_edu          0.174296
Gas_Flow_edu                     0.140723
Cooling_Water_Temperature_edu    0.131259
Vacuum_Level_edu                 0.107293
Process_Time_edu                 0.091585
RF_Power_edu                     0.075310
Chamber_Pressure_edu             0.059280
Vibration_edu                    0.000000
주요센서_B                           0.000000
dtype: float64

## 10단계. 오늘의 분석을 한 문장으로 정리하기

> 모델의 정확도는 90%를 크게 넘었지만, '모두 정상'이라고 예측해도 90% 이상의 정확도가 나온다.
> 따라서 정확도만으로는 모델의 성능을 판단하기 어렵고, 혼동행렬로 실제 이상을 얼마나 잡아냈는지
> 함께 봐야 한다. 또한 변수 중요도가 높다고 해서 그 센서가 불량의 실제 원인이라고 단정할 수 없다.